# Surface Drones | Terrorizer research notebook

In [1]:
from pathlib import Path
from contextlib import contextmanager
import os
import sys
import re
import json
import hashlib
import zipfile
import importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve()
SOURCE_ZIP = PROJECT_DIR / 'Terrorizer-main.zip'
GOLD_CSV = PROJECT_DIR / 'PATSTAT_500_label_datav4.csv'
SOURCE_DIR = PROJECT_DIR / 'terrorizer_original_source'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'terrorizer_original_source_pilot'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_MAX_NAMES = 120
SEED = 42
MODEL_ID = 'Vsevolod/company-names-similarity-sentence-transformer'
MODEL_DEVICE = 'cpu'

WEIGHT_SAME_TOKEN = 0.30
WEIGHT_FIRST_TOKEN = 0.20
WEIGHT_SIMILARITY = 1.00
MATCH_THRESHOLD = 0.87
BRIDGENESS_THRESHOLD = 0.10
LOUVAIN_RESOLUTION = 1.0

assert 2 <= PILOT_MAX_NAMES <= 250
print('Pilot sample:', PILOT_MAX_NAMES, '| random seed:', SEED)
print('Running on the Gold Standard CSV; no PATSTAT PROD queries.')

Pilot sample: 120 | random seed: 42
Running on the Gold Standard CSV; no PATSTAT PROD queries.


In [2]:
# Import only the modules needed for this study.
assert SOURCE_ZIP.is_file(), f'Missing source ZIP: {SOURCE_ZIP.name}'
source_files = (
    'README.md', 'bridges.py', 'preprocessing_class_terrorizer.py',
    'vectorizer_class2.py', 'process_network_terrorizer.py',
    'knowledge_augementation_class.py',
)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    names = [name for name in archive.namelist() if not name.endswith('/')]
    for filename in source_files:
        matches = [name for name in names if name.endswith('/' + filename)]
        assert len(matches) == 1, f'Expected one {filename} in {SOURCE_ZIP.name}'
        destination = SOURCE_DIR / filename
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(matches[0]))

source_sha256 = hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest()
print('Source files extracted to:', SOURCE_DIR)
print('Source archive SHA256:', source_sha256)

Source files extracted to: /home/jovyan/Surface_Drones_Project/terrorizer_original_source
Source archive SHA256: 691b36a66d3e3dd194bfeb02fbec78fd79db70500da82a42324c31d466608e94


In [3]:
# The upstream modules import some packages even when particular features are unused.
requirements = {
    'cleanco': 'cleanco', 'toolz': 'toolz', 'nltk': 'nltk',
    'gensim': 'gensim', 'tldextract': 'tldextract', 'swifter': 'swifter',
    'sentence_transformers': 'sentence-transformers', 'networkx': 'networkx',
    'numba': 'numba', 'atpbar': 'atpbar', 'mantichora': 'mantichora',
    'pandarallel': 'pandarallel', 'optuna': 'optuna', 'joblib': 'joblib',
    'sklearn': 'scikit-learn', 'torch': 'torch', 'numpy': 'numpy',
    'pandas': 'pandas',
}
missing = [pkg for module, pkg in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('Install the missing packages in a separate Jupyter cell:')
    print('%pip install ' + ' '.join(missing))
    print('Restart the kernel and run all cells again.')
    raise ImportError('Missing dependencies: ' + ', '.join(missing))
print('Package availability check passed.')

Package availability check passed.


In [4]:
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
from preprocessing_class_terrorizer import NamesPreprocessing
from process_network_terrorizer import terrorizer_matching_stage, process_network
from sentence_transformers import SentenceTransformer
from cleanco import basename

# The upstream constructor downloads NLTK resources; preprocess() does not need it.
preprocessor = NamesPreprocessing.__new__(NamesPreprocessing)
model = SentenceTransformer(MODEL_ID, device=MODEL_DEVICE)
print('Source modules and embedding model loaded.')

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Source modules and embedding model loaded.


In [5]:
# Known differences from the upstream repository, recorded for reproducibility.
source_audit = pd.DataFrame([
    ('Installation', 'The supplied ZIP has modules but no setup.py or pyproject.toml.'),
    ('Vectorizer', 'Source get_vector() does not return one vector per input name; use batch model.encode().'),
    ('Preprocessing', 'Bypass __init__ downloads; call the source preprocess() function.'),
    ('Knowledge augmentation', 'External knowledge_base_duckduck24v2.csv is absent; this stage is skipped.'),
    ('Matching', 'Matching and community detection use the original source classes.'),
    ('Thresholds', 'Working values are exploratory, not validated production settings.'),
], columns=['component', 'note'])
display(source_audit)
source_audit.to_csv(OUTPUT_DIR / 'source_audit.csv', index=False)

,component,note
0,Installation,The supplied ZIP has modules but no setup.py o...
1,Vectorizer,Source get_vector() does not return one vector...
2,Preprocessing,Bypass __init__ downloads; call the source pre...
3,Knowledge augmentation,External knowledge_base_duckduck24v2.csv is ab...
4,Matching,Matching and community detection use the origi...
5,Thresholds,"Working values are exploratory, not validated ..."


## 1. Shared functions

The three experiments use the same preprocessing, embedding model, matching parameters and pairwise evaluation. Defining them once keeps the results comparable and avoids repeating the same calculations.

In [6]:
LEGAL_FORMS = {
    'INC', 'INCORPORATED', 'CORP', 'CORPORATION', 'CO', 'COMPANY',
    'LTD', 'LIMITED', 'LLC', 'LP', 'PLC', 'GMBH', 'AG', 'KG', 'MBH',
    'SE', 'SA', 'SAS', 'SPA', 'SRL', 'BV', 'NV', 'AB', 'AS', 'ASA',
    'OY', 'OYJ', 'KK', 'KABUSHIKI', 'KAISHA', 'LTDA', 'PTY',
    'THE', 'AKTIENGESELLSCHAFT', 'GESELLSCHAFT',
}

def clean_with_dictionary(name):
    words = re.sub(r'[^A-Z0-9 ]', ' ', str(name).upper()).split()
    while words and words[-1] in LEGAL_FORMS:
        words.pop()
    while words and words[0] in LEGAL_FORMS:
        words.pop(0)
    return ' '.join(words)

def clean_with_cleanco(name):
    original = str(name).strip()
    cleaned = re.sub(r'[^A-Z0-9]+', ' ', basename(original).upper()).strip()
    fallback = re.sub(r'[^A-Z0-9]+', ' ', original.upper()).strip()
    return cleaned or fallback or original.upper()

def pairwise_metrics(data, cluster_column, method):
    reference = data['Group'].to_numpy()
    predicted = data[cluster_column].to_numpy()
    i, j = np.triu_indices(len(data), k=1)
    same_ref = reference[i] == reference[j]
    same_pred = predicted[i] == predicted[j]
    tp = int((same_ref & same_pred).sum())
    fp = int((~same_ref & same_pred).sum())
    fn = int((same_ref & ~same_pred).sum())
    tn = int((~same_ref & ~same_pred).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    assert tp + fp + fn + tn == len(data) * (len(data) - 1) // 2
    return dict(method=method, names=len(data), clusters=data[cluster_column].nunique(),
                TP=tp, FP=fp, FN=fn, TN=tn,
                precision=precision, recall=recall, F1=f1)

@contextmanager
def temporary_cwd(directory):
    previous = Path.cwd()
    try:
        os.chdir(directory)
        yield
    finally:
        os.chdir(previous)

def flatten_groups(results):
    groups = []
    for item in results.values():
        if isinstance(item, set):
            groups.append(item)
        elif isinstance(item, (list, tuple)):
            if not all(isinstance(part, set) for part in item):
                raise TypeError('Unexpected community member type')
            groups.extend(item)
        else:
            raise TypeError(f'Unexpected community type: {type(item)}')
    return groups

def prepare_names(data):
    frame = pd.DataFrame({'original_name': data['original_name'].astype(str).tolist()})
    normal = preprocessor.preprocess(frame, default_column='original_name')
    normal['name'] = normal['original_name']
    normal['name_normalized'] = normal['name_normalized'].fillna('').astype(str)
    empty = normal['name_normalized'].str.strip().eq('')
    if empty.any():
        normal.loc[empty, 'name_normalized'] = normal.loc[empty, 'original_name'].str.lower()
    assert len(normal) == len(data) and normal['name_normalized'].str.strip().ne('').all()
    return normal

def run_terrorizer(data, run_dir):
    # This matching implementation compares name pairs; keep each run small.
    assert 2 <= len(data) <= 250
    run_dir.mkdir(parents=True, exist_ok=True)
    normal = prepare_names(data)
    vectors = model.encode(normal['name_normalized'].tolist(),
                           convert_to_numpy=True, show_progress_bar=True,
                           batch_size=16, normalize_embeddings=False)
    assert vectors.ndim == 2 and vectors.shape[0] == len(data)
    assert np.isfinite(vectors).all()
    np.save(run_dir / 'embeddings.npy', vectors)
    with temporary_cwd(run_dir):
        matcher = terrorizer_matching_stage(
            data=normal[['original_name', 'name_normalized', 'name']].copy(),
            vectors=vectors, weight_same_t=WEIGHT_SAME_TOKEN,
            weight_first_token=WEIGHT_FIRST_TOKEN,
            weight_similarity=WEIGHT_SIMILARITY, weight_in_domain=0,
            weight_token_text=0,
        )
        scores = matcher.main()
        assert len(scores) == len(data)
        network = process_network(results_normal=scores,
                                  resolution=LOUVAIN_RESOLUTION,
                                  match_threshold=MATCH_THRESHOLD,
                                  bridgeness_threshold=BRIDGENESS_THRESHOLD)
        communities = network.main()
    assignments = {}
    for cluster_id, nodes in enumerate(flatten_groups(communities)):
        for node in nodes:
            index = int(node)
            assert index not in assignments, 'Duplicate cluster assignment'
            assignments[index] = cluster_id
    assert set(assignments) == set(range(len(data))), 'Missing cluster assignments'
    result = data.copy().reset_index(drop=True)
    result['terrorizer_cluster'] = result['row_id'].map(assignments)
    representatives = result.groupby('terrorizer_cluster')['original_name'].apply(
        lambda names: sorted(names.tolist(), key=lambda x: (len(x), x))[0]).to_dict()
    result['representative_name'] = result['terrorizer_cluster'].map(representatives)
    return result, normal, network

print('Shared preprocessing, matching and evaluation functions ready.')

Shared preprocessing, matching and evaluation functions ready.


## 2. Gold Standard pilot (120 names)

The pilot intentionally includes multiple names from some reference groups, then fills remaining places from the Gold Standard. All three methods are evaluated on **exactly the same sample**.

In [7]:
all_names = pd.read_csv(GOLD_CSV)[['Group', 'comp']].dropna().drop_duplicates('comp')
rng = np.random.default_rng(SEED)
groups = [group for _, group in all_names.groupby('Group') if len(group) >= 2]
rng.shuffle(groups)
chosen = []
max_groups = min(len(groups), max(12, PILOT_MAX_NAMES // 3))
for group in groups[:max_groups]:
    count = min(len(group), 3, PILOT_MAX_NAMES - len(chosen))
    if count <= 0:
        break
    chosen.extend(group.sample(n=count, random_state=SEED)['comp'].tolist())
picked = all_names[all_names['comp'].isin(chosen)].copy()
if len(picked) < PILOT_MAX_NAMES:
    others = all_names[~all_names['comp'].isin(chosen)]
    picked = pd.concat([
        picked,
        others.sample(n=min(PILOT_MAX_NAMES - len(picked), len(others)), random_state=SEED),
    ], ignore_index=True)
sample = picked.head(PILOT_MAX_NAMES).rename(columns={'comp': 'original_name'}).reset_index(drop=True)
sample.insert(0, 'row_id', range(len(sample)))
assert sample['original_name'].is_unique and len(sample) == PILOT_MAX_NAMES
sample.to_csv(OUTPUT_DIR / 'sample_gold.csv', index=False)
print('Pilot names:', len(sample), '| reference groups:', sample['Group'].nunique())
display(sample.head(12))

Pilot names: 120 | reference groups: 46


,row_id,Group,original_name
0,0,4,TOSHIBA PERSONAL SYSTEM ENGINEERING CORPORATION
1,1,4,TOSHIBA AVE
2,2,4,TOSHIBA PLANT KENSETSU
3,3,11,SANOFI PASTEUR BIOLOGICS
4,4,11,SANOFI BIO INGREDIENTS
5,5,11,SANOFI PHARMACEUTICALS
6,6,12,STMICROELECTRONICS SAS (MAROC)
7,7,12,STMICROELECTRONICS APPLICATION
8,8,12,STMICROELECTRONICS (SHENZHEN) R&D
9,9,13,INEOS VINYLS


In [8]:
pilot_clusters, pilot_normal, pilot_network = run_terrorizer(sample, OUTPUT_DIR / 'pilot_matching')
pilot_clusters.to_csv(OUTPUT_DIR / 'terrorizer_clusters_gold.csv', index=False)
print('Pilot clusters:', pilot_clusters['terrorizer_cluster'].nunique())
display(pilot_clusters[['Group', 'original_name', 'terrorizer_cluster', 'representative_name']].head(20))

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Communities detection started
[{0, 1, 2, 68, 111}, {114, 3, 4, 5}, {8, 6, 7}, {9, 10, 11}, {12, 14}, {13}, {17, 18, 15}, {16, 19}, {20, 21, 22}, {24, 70, 23}, {25, 26, 27}, {29, 30, 31}, {32, 33, 28}, {34, 35, 36}, {37, 38, 39}, {40, 41}, {42}, {43}, {44}, {45, 46}, {48, 49, 47}, {50, 51, 52}, {98, 100, 53, 102, 104}, {54, 55}, {56, 57, 58}, {59, 60, 61}, {64, 62, 63}, {65, 66, 67}, {69}, {71}, {72}, {73, 74, 75}, {76, 77, 78}, {80, 81, 79}, {82, 83, 84}, {85, 86, 87}, {88, 91}, {89, 90}, {92}, {93}, {94}, {95}, {96, 97}, {99, 101}, {105, 103}, {106, 107, 108}, {109, 110}, {112}, {113}, {115}, {116}, {117}, {118}, {119}]
Pilot clusters: 54


,Group,original_name,terrorizer_cluster,representative_name
0,4,TOSHIBA PERSONAL SYSTEM ENGINEERING CORPORATION,0,TOSHIBA AVE
1,4,TOSHIBA AVE,0,TOSHIBA AVE
2,4,TOSHIBA PLANT KENSETSU,0,TOSHIBA AVE
3,11,SANOFI PASTEUR BIOLOGICS,1,SANOFI BIO INGREDIENTS
4,11,SANOFI BIO INGREDIENTS,1,SANOFI BIO INGREDIENTS
5,11,SANOFI PHARMACEUTICALS,1,SANOFI BIO INGREDIENTS
6,12,STMICROELECTRONICS SAS (MAROC),2,STMICROELECTRONICS APPLICATION
7,12,STMICROELECTRONICS APPLICATION,2,STMICROELECTRONICS APPLICATION
8,12,STMICROELECTRONICS (SHENZHEN) R&D,2,STMICROELECTRONICS APPLICATION
9,13,INEOS VINYLS,3,INEOS VINYLS


In [9]:
pilot_rules = pilot_clusters.copy()
pilot_rules['dictionary_key'] = pilot_rules['original_name'].apply(clean_with_dictionary)
pilot_rules['cleanco_key'] = pilot_rules['original_name'].apply(clean_with_cleanco)
assert pilot_rules[['dictionary_key', 'cleanco_key']].ne('').all().all()

pilot_comparison = pd.DataFrame([
    pairwise_metrics(pilot_rules, 'dictionary_key', 'Dictionary'),
    pairwise_metrics(pilot_rules, 'cleanco_key', 'Cleanco'),
    pairwise_metrics(pilot_rules, 'terrorizer_cluster', 'Terrorizer (Pilot)'),
]).sort_values('F1', ascending=False).reset_index(drop=True)
display(pilot_comparison.round(4))
pilot_comparison.to_csv(OUTPUT_DIR / 'three_method_comparison.csv', index=False)
pilot_rules[['row_id', 'Group', 'original_name', 'dictionary_key']].to_csv(
    OUTPUT_DIR / 'dictionary_clusters_gold.csv', index=False)
pilot_rules[['row_id', 'Group', 'original_name', 'cleanco_key']].to_csv(
    OUTPUT_DIR / 'cleanco_clusters_gold.csv', index=False)
# Keep a separate Terrorizer result for compatibility with previous reports.
pilot_comparison[pilot_comparison['method'] == 'Terrorizer (Pilot)'].to_csv(
    OUTPUT_DIR / 'sample_pairwise_evaluation.csv', index=False)

,method,names,clusters,TP,FP,FN,TN,precision,recall,F1
0,Terrorizer (Pilot),120,54,92,11,19,7018,0.8932,0.8288,0.8598
1,Cleanco,120,118,2,0,109,7029,1.0000,0.0180,0.0354
2,Dictionary,120,119,1,0,110,7029,1.0000,0.0090,0.0179


In [10]:
pilot_review = pilot_clusters.groupby('terrorizer_cluster').agg(
    n_names=('original_name', 'size'),
    reference_groups=('Group', 'nunique'),
    names=('original_name', lambda names: ' | '.join(names.head(8))),
).reset_index().sort_values(['n_names', 'terrorizer_cluster'], ascending=[False, True])
pilot_review.to_csv(OUTPUT_DIR / 'clusters_for_review.csv', index=False)
pilot_risky = pilot_review[pilot_review['reference_groups'] > 1]
print('Pilot clusters joining different reference groups:', len(pilot_risky))
display(pilot_review.head(15))
pilot_clusters[pilot_clusters['terrorizer_cluster'].isin(pilot_risky['terrorizer_cluster'])].to_csv(
    OUTPUT_DIR / 'cross_group_merges_for_review.csv', index=False)

Pilot clusters joining different reference groups: 2


,terrorizer_cluster,n_names,reference_groups,names
0,0,5,2,TOSHIBA PERSONAL SYSTEM ENGINEERING CORPORATIO...
22,22,5,3,VALEO SIEMENS EAUTOMOTIVE | VALEO SYSTEMES D'E...
1,1,4,1,SANOFI PASTEUR BIOLOGICS | SANOFI BIO INGREDIE...
2,2,3,1,STMICROELECTRONICS SAS (MAROC) | STMICROELECTR...
3,3,3,1,INEOS VINYLS | INEOS NOVA INTERNATIONAL | INEO...
6,6,3,1,AMAZONEN-WERKE H. DREYER | AMAZONEN-WERKE | AM...
8,8,3,1,EXXONMOBIL UPSTREAM RESEARCH COMPANY COPR-URC ...
9,9,3,1,MCKESSON AUTOMATION | MCKESSON AUTOMATION SYST...
10,10,3,1,INTEL COPORATION | INTEL CORPORATION (INTEL) |...
11,11,3,1,"APPLERA CORPORATION, APPLIED BIOSYSTEMS GROUP ..."


## 3. Independent Gold Standard holdout (200 names)

The holdout excludes every reference group represented in the pilot. It tests whether the same settings work on **previously unseen groups**. Its pair distribution is not necessarily representative of the complete dataset, so inspect the group-size balance before interpreting F1.

In [11]:
full_gold = pd.read_csv(GOLD_CSV)[['Group', 'comp']].dropna().drop_duplicates('comp')
pilot_groups = set(sample['Group'].astype(str))
available = full_gold[~full_gold['Group'].astype(str).isin(pilot_groups)]
assert len(available) >= 200
holdout = available.sample(n=200, random_state=43).reset_index(drop=True)
holdout = holdout.rename(columns={'comp': 'original_name'})
holdout.insert(0, 'row_id', range(len(holdout)))
assert set(holdout['Group'].astype(str)).isdisjoint(pilot_groups)
holdout.to_csv(OUTPUT_DIR / 'holdout_gold_200.csv', index=False)

group_sizes = holdout.groupby('Group').size().sort_values(ascending=False)
positive_pairs = group_sizes.mul(group_sizes - 1).floordiv(2)
positive_total = int(positive_pairs.sum())
print('Holdout names:', len(holdout), '| reference groups:', holdout['Group'].nunique())
print('Positive reference pairs:', positive_total)
print('Share of positive pairs from five largest groups:',
      f'{positive_pairs.head(5).sum() / positive_total:.1%}' if positive_total else 'not defined')
display(pd.DataFrame({'names': group_sizes, 'positive_pairs': positive_pairs}).head(10))

Holdout names: 200 | reference groups: 98
Positive reference pairs: 388
Share of positive pairs from five largest groups: 70.4%


,names,positive_pairs
Group,,
212,18,153
19,9,36
340,8,28
292,8,28
290,8,28
9,7,21
486,6,15
7,6,15
207,5,10


In [12]:
holdout_rules = holdout.copy()
holdout_rules['dictionary_key'] = holdout_rules['original_name'].apply(clean_with_dictionary)
holdout_rules['cleanco_key'] = holdout_rules['original_name'].apply(clean_with_cleanco)
assert holdout_rules[['dictionary_key', 'cleanco_key']].ne('').all().all()
holdout_rules.to_csv(OUTPUT_DIR / 'holdout_rules_200.csv', index=False)

holdout_clusters, holdout_normal, holdout_network = run_terrorizer(
    holdout, OUTPUT_DIR / 'holdout_200_matching')
holdout_clusters.to_csv(OUTPUT_DIR / 'holdout_terrorizer_clusters_200.csv', index=False)
print('Holdout Terrorizer clusters:', holdout_clusters['terrorizer_cluster'].nunique())

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Communities detection started
[{0}, {2}, {6}, {8}, {10}, {14}, {24, 23}, {25, 121}, {26}, {194, 68, 69, 76, 142, 145, 20, 93, 29, 94, 160, 161, 102, 169, 106, 171}, {31}, {32}, {33}, {35, 47}, {36}, {37}, {38}, {126, 39}, {41}, {42, 173}, {43}, {64, 45}, {46}, {50, 116, 7, 57, 63}, {51, 141}, {52}, {55}, {65}, {66}, {70}, {71}, {176, 108, 34, 56, 73, 74, 28}, {16, 77}, {78}, {79}, {80}, {82}, {83, 101}, {21, 86}, {88}, {90}, {91, 174}, {92}, {13, 95}, {96, 17, 117}, {153, 133, 103}, {104, 3}, {105}, {49, 182, 87, 152, 107}, {110}, {113}, {115}, {119}, {122}, {1, 130, 12, 4, 124, 125, 60}, {127}, {128}, {131}, {62, 146, 59, 134}, {135}, {136}, {67, 100, 197, 72, 137, 138, 147, 120, 155}, {178, 139, 111}, {140}, {143}, {144, 30}, {149}, {150}, {154}, {156, 132, 180, 22, 188}, {48, 53, 168, 157, 15}, {158}, {159}, {129, 162, 19, 114, 98, 148, 58}, {97, 163, 123, 109, 112, 18, 84, 27}, {164}, {165}, {166}, {170}, {99, 167, 40, 75, 172, 44, 151, 89, 61}, {175}, {177}, {179}, {183}, {184, 9,

In [13]:
holdout_comparison = pd.DataFrame([
    pairwise_metrics(holdout_rules, 'dictionary_key', 'Dictionary'),
    pairwise_metrics(holdout_rules, 'cleanco_key', 'Cleanco'),
    pairwise_metrics(holdout_clusters, 'terrorizer_cluster', 'Terrorizer (Holdout)'),
]).sort_values('F1', ascending=False).reset_index(drop=True)
assert (holdout_comparison['TP'] + holdout_comparison['FN'] == positive_total).all()
display(holdout_comparison.round(4))
holdout_comparison.to_csv(OUTPUT_DIR / 'three_method_comparison_200.csv', index=False)
holdout_comparison[holdout_comparison['method'] != 'Terrorizer (Holdout)'].to_csv(
    OUTPUT_DIR / 'holdout_rules_evaluation_200.csv', index=False)

,method,names,clusters,TP,FP,FN,TN,precision,recall,F1
0,Terrorizer (Holdout),200,95,342,32,46,19480,0.9144,0.8814,0.8976
1,Dictionary,200,198,2,0,386,19512,1.0000,0.0052,0.0103
2,Cleanco,200,198,2,0,386,19512,1.0000,0.0052,0.0103


In [14]:
# A wrong merge creates a false-positive pair; a missed merge creates a false-negative pair.
names = holdout_clusters['original_name'].to_numpy()
truth = holdout_clusters['Group'].to_numpy()
predicted = holdout_clusters['terrorizer_cluster'].to_numpy()
i, j = np.triu_indices(len(holdout_clusters), k=1)
actual_same = truth[i] == truth[j]
predicted_same = predicted[i] == predicted[j]

def error_pairs(mask):
    a, b = i[mask], j[mask]
    return pd.DataFrame({'name_1': names[a], 'name_2': names[b],
                         'reference_1': truth[a], 'reference_2': truth[b],
                         'predicted_1': predicted[a], 'predicted_2': predicted[b]})

false_positives = error_pairs(~actual_same & predicted_same)
false_negatives = error_pairs(actual_same & ~predicted_same)
row = holdout_comparison.set_index('method').loc['Terrorizer (Holdout)']
assert len(false_positives) == row['FP'] and len(false_negatives) == row['FN']
false_positives.to_csv(OUTPUT_DIR / 'holdout_false_positives_200.csv', index=False)
false_negatives.to_csv(OUTPUT_DIR / 'holdout_false_negatives_200.csv', index=False)
print('False positive pairs:', len(false_positives), '| false negative pairs:', len(false_negatives))
display(false_positives.head(10))
display(false_negatives.head(10))

False positive pairs: 32 | false negative pairs: 46


,name_1,name_2,reference_1,reference_2,predicted_1,predicted_2
0,PIRELLI CABLES (2000) LIMITED,MICHELIN & CIE (COMPAGNIE GENERALE DES ETABLIS...,227,472,87,87
1,PIRELLI CABLES (2000) LIMITED,MICHELIN RECHERCHE ET TECNIQUE,227,472,87,87
2,SAFRAN AERO BOOSTERS,AIRBUS-SAFRAN LAUNCHERS,387,388,23,23
3,SAFRAN AERO BOOSTERS,AIRBUS SAFRAN LAUNCHERS,387,388,23,23
4,NOKIA KAAPELI,NOKIA SIEMENS NETWORKS ASSET MANAGEMENT,1,2,43,43
5,THALES ANTENNAS LIMITED,ATELIERS MECANIQUES DE FONTVIEILLE LE THALES,486,488,73,73
6,OCEAN PHOENIX HOLDINGS,PROJECT PHOENIX,163,167,38,38
7,I.P.A. S.P.A. - INDUSTRIA PREFABBRICATI AFFINI,F.A.E. INDUSTRIA ALLOGGI PREFABBRICATI,421,410,65,65
8,SAFRAN CERAMICS,AIRBUS-SAFRAN LAUNCHERS,387,388,23,23
9,SAFRAN CERAMICS,AIRBUS SAFRAN LAUNCHERS,387,388,23,23


,name_1,name_2,reference_1,reference_2,predicted_1,predicted_2
0,NOKIA KAAPELI,SCANINTER NOKIA,1,1,43,75
1,SIEMENS NETWORK CONVERGENCE,SIEMENS SCHIENENFAHRZEUGTECHNIK,212,212,9,12
2,SIEMENS NETWORK CONVERGENCE,SIEMENS AXIVA,212,212,9,52
3,HONEYWELL LUCIFER,HONEYWELL-BRAUKMANN,292,292,31,93
4,SIEMENS VAI METALS TECHNOLOGIES,SIEMENS SCHIENENFAHRZEUGTECHNIK,212,212,9,12
5,SIEMENS VAI METALS TECHNOLOGIES,SIEMENS AXIVA,212,212,9,52
6,SIEMENS SCHIENENFAHRZEUGTECHNIK,SIEMENS NIXDORF INFORMATIONSSYSTEME,212,212,12,9
7,SIEMENS SCHIENENFAHRZEUGTECHNIK,SIEMENS TRANSIT TELEMATIC SYSTEMS,212,212,12,9
8,SIEMENS SCHIENENFAHRZEUGTECHNIK,SIEMENS VDO AUTOMOTIVE CORPORATION,212,212,12,9
9,SIEMENS SCHIENENFAHRZEUGTECHNIK,SIEMENS AKTIENGESELLSCHAFT OSTERREICH,212,212,12,9


In [15]:
mixed_clusters = holdout_clusters.groupby('terrorizer_cluster').agg(
    names_count=('original_name', 'size'), reference_groups=('Group', 'nunique'),
    group_ids=('Group', lambda s: ', '.join(sorted(set(map(str, s))))),
    names=('original_name', lambda s: ' | '.join(s.head(6))),
).reset_index().query('reference_groups > 1')
split_groups = holdout_clusters.groupby('Group').agg(
    names_count=('original_name', 'size'), predicted_clusters=('terrorizer_cluster', 'nunique'),
    cluster_ids=('terrorizer_cluster', lambda s: ', '.join(sorted(set(map(str, s))))),
    names=('original_name', lambda s: ' | '.join(s.head(6))),
).reset_index().query('predicted_clusters > 1')
print('Mixed predicted clusters:', len(mixed_clusters),
      '| reference groups split across clusters:', len(split_groups))
display(mixed_clusters.head(15))
display(split_groups.head(15))
mixed_clusters.to_csv(OUTPUT_DIR / 'holdout_mixed_clusters_200.csv', index=False)
split_groups.to_csv(OUTPUT_DIR / 'holdout_split_groups_200.csv', index=False)

Mixed predicted clusters: 8 | reference groups split across clusters: 4


,terrorizer_cluster,names_count,reference_groups,group_ids,names
23,23,5,2,"387, 388",SAFRAN AERO BOOSTERS | SAFRAN CERAMICS | AIRBU...
38,38,2,2,"163, 167",OCEAN PHOENIX HOLDINGS | PROJECT PHOENIX
43,43,2,2,"1, 2",NOKIA KAAPELI | NOKIA SIEMENS NETWORKS ASSET M...
61,61,9,2,"340, 342",SCHLUMBERGER TECHNOLOGIES LIMITED | N. SCHLUMB...
65,65,2,2,"410, 421",I.P.A. S.P.A. - INDUSTRIA PREFABBRICATI AFFINI...
73,73,7,2,"486, 488","THALES ANTENNAS LIMITED | THALES, INTELLECTUAL..."
87,87,6,2,"227, 472",PIRELLI CABLES (2000) LIMITED | PIRELLI KABEL ...
91,91,2,2,"425, 426",F.I.M.A.C. FABBRICA ITALIANA MACCHINE ARIA COM...


,Group,names_count,predicted_clusters,cluster_ids,names
0,1,2,2,"43, 75",NOKIA KAAPELI | SCANINTER NOKIA
2,7,6,2,"48, 50",TOYOTA MOTOR COMPANY | TOYOTA TSUSHO AMERICA |...
33,212,18,3,"12, 52, 9",SIEMENS NETWORK CONVERGENCE | SIEMENS VAI META...
53,292,8,2,"31, 93",HONEYWELL LUCIFER | HONEYWELL AEROSPATIALE | H...


## 4. Reproducibility and remaining validation

These CSV files contain **Gold Standard sample results**, not a validated sector-wide disambiguation. The holdout has no reference groups in common with the pilot, but the sample distribution and threshold choices still need to be discussed before reporting performance.

Use the main Step 2 notebook for the Surface Drones applicant analysis. This research notebook does not require or read `raw_family_applicants.csv`.

In [16]:
manifest = {
    'source_zip_sha256': source_sha256,
    'source': 'Terrorizer original repository ZIP',
    'experiment': 'Source-based Terrorizer pilot and independent Gold Standard holdout',
    'upstream_unmodified': False,
    'changes': [
        'Full-name batch embeddings with the named source model',
        'Source preprocessing called without NLTK download side effects',
        'External knowledge augmentation skipped (missing external CSV)',
    ],
    'model': MODEL_ID,
    'seed_pilot': SEED,
    'pilot_names': len(sample),
    'holdout_names': len(holdout),
    'parameters': {
        'same_token': WEIGHT_SAME_TOKEN, 'first_token': WEIGHT_FIRST_TOKEN,
        'similarity': WEIGHT_SIMILARITY, 'match_threshold': MATCH_THRESHOLD,
        'bridgeness_threshold': BRIDGENESS_THRESHOLD,
        'resolution': LOUVAIN_RESOLUTION,
    },
    'final_industry_analysis_ready': False,
}
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Outputs:', OUTPUT_DIR)
print('Gold Standard pilot and holdout only; do not present these scores as sector-wide accuracy.')

Outputs: /home/jovyan/Surface_Drones_Project/outputs/terrorizer_original_source_pilot
Gold Standard pilot and holdout only; do not present these scores as sector-wide accuracy.
